# 06 - Train YOLOv8 Classification

This notebook trains a YOLOv8 classification model on the prepared YOLO folder dataset.

Training plan:
- Model: `yolov8n-cls.pt`
- Epochs: 12 fixed
- No early stopping: `patience=1000`, so it will not stop early within 12 epochs
- Image size: 224
- Batch size: 32
- Dataset: `data/yolo_cls_balanced`

Important difference from PyTorch models:
- PyTorch used `WeightedRandomSampler`.
- YOLO reads folders, so train data was balanced using symlinks.
- Validation and test splits remain natural/unbalanced.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/backup/Intern/combine_skindiseaseclassifier_devraj')
EXPECTED_PYTHON = str(PROJECT_ROOT / '.venv' / 'bin' / 'python')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

if sys.executable != EXPECTED_PYTHON:
    raise RuntimeError(
        'Wrong notebook kernel selected. Expected: ' + EXPECTED_PYTHON + '\n'
        'Current: ' + sys.executable + '\n'
        'Please switch kernel to: Combine Skin GPU (.venv)'
    )

Project root: /backup/Intern/combine_skindiseaseclassifier_devraj
Python: /backup/Intern/combine_skindiseaseclassifier_devraj/.venv/bin/python


In [2]:
from datetime import datetime
import json
import subprocess

import torch
from ultralytics import YOLO

DATA_DIR = PROJECT_ROOT / 'data' / 'yolo_cls_balanced'
RUNS_DIR = PROJECT_ROOT / 'training_outputs' / 'yolo'
REPORT_MD_DIR = PROJECT_ROOT / 'markdown_reports'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'yolo_training'

RUNS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Ultralytics import: OK')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('YOLO data:', DATA_DIR)

Ultralytics import: OK
Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3
YOLO data: /backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced


## Check YOLO Dataset

Expected folder structure:

```text
data/yolo_cls_balanced/
  train/class_name/images
  val/class_name/images
  test/class_name/images
```

In [3]:
for split in ['train', 'val', 'test']:
    split_dir = DATA_DIR / split
    class_counts = {}
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        count = sum(1 for p in class_dir.rglob('*') if p.is_symlink() or p.is_file())
        class_counts[class_dir.name] = count
    print('\n', split, 'total:', sum(class_counts.values()))
    for name, count in class_counts.items():
        print(f'  {name}: {count}')


 train total: 24178
  acne_vulgaris: 1727
  atopic_dermatitis: 1727
  basal_cell_carcinoma: 1727
  contact_dermatitis: 1727
  drug_eruptions: 1727
  folliculitis: 1727
  fungal_nail_infections: 1727
  lupus_related_skin_lesions: 1727
  melanoma: 1727
  plaque_psoriasis: 1727
  seborrheic_dermatitis: 1727
  tinea_corporis: 1727
  vitiligo: 1727
  warts: 1727

 val total: 5478
  acne_vulgaris: 370
  atopic_dermatitis: 295
  basal_cell_carcinoma: 910
  contact_dermatitis: 227
  drug_eruptions: 239
  folliculitis: 177
  fungal_nail_infections: 180
  lupus_related_skin_lesions: 193
  melanoma: 1364
  plaque_psoriasis: 502
  seborrheic_dermatitis: 180
  tinea_corporis: 290
  vitiligo: 251
  warts: 300

 test total: 5474
  acne_vulgaris: 370
  atopic_dermatitis: 294
  basal_cell_carcinoma: 910
  contact_dermatitis: 227
  drug_eruptions: 238
  folliculitis: 176
  fungal_nail_infections: 180
  lupus_related_skin_lesions: 193
  melanoma: 1364
  plaque_psoriasis: 502
  seborrheic_dermatitis: 180

## Configuration

Use fixed 12 epochs. `patience=1000` prevents YOLO early stopping during this 12-epoch run.

In [4]:
REQUIRE_GPU = True
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('CUDA/GPU is not available. Do not train YOLO on CPU.')

MODEL_FILE = 'yolov8n-cls.pt'
IMAGE_SIZE = 224
EPOCHS = 12
BATCH_SIZE = 32
WORKERS = 0
DEVICE = 0
PATIENCE = 1000  # effectively disables early stopping for a 12-epoch run
RUN_NAME = f'yolov8n_cls_fixed_12ep_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

config = {
    'model_file': MODEL_FILE,
    'data_dir': str(DATA_DIR),
    'image_size': IMAGE_SIZE,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'workers': WORKERS,
    'device': DEVICE,
    'patience': PATIENCE,
    'early_stopping': False,
    'run_name': RUN_NAME,
}
config

{'model_file': 'yolov8n-cls.pt',
 'data_dir': '/backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced',
 'image_size': 224,
 'epochs': 12,
 'batch_size': 32,
 'workers': 0,
 'device': 0,
 'patience': 1000,
 'early_stopping': False,
 'run_name': 'yolov8n_cls_fixed_12ep_20260701_121336'}

## Train YOLOv8 Classification

The first run may download `yolov8n-cls.pt` pretrained weights.

In [5]:
model = YOLO(MODEL_FILE)

train_results = model.train(
    data=str(DATA_DIR),
    imgsz=IMAGE_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,
    pretrained=True,
    plots=True,
    seed=42,
)

RUN_DIR = RUNS_DIR / RUN_NAME
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
LAST_MODEL = RUN_DIR / 'weights' / 'last.pt'

print('Run folder:', RUN_DIR)
print('Best model:', BEST_MODEL)
print('Last model:', LAST_MODEL)

Ultralytics 8.4.84 🚀 Python-3.10.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81090MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=12, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_cls_fixed_12ep_20260701_121336

## Validate Best Model On Test Split

YOLO already validates during training on `val`. This cell evaluates the saved best checkpoint on the natural `test` split.

In [6]:
best_model = YOLO(str(BEST_MODEL))

test_results = best_model.val(
    data=str(DATA_DIR),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)

metrics = {
    'model': 'yolov8n-cls',
    'run_name': RUN_NAME,
    'run_dir': str(RUN_DIR),
    'best_model': str(BEST_MODEL),
    'test_top1_acc': float(getattr(test_results, 'top1', 0.0)),
    'test_top5_acc': float(getattr(test_results, 'top5', 0.0)),
}

metrics_path = REPORT_DIR / f'{RUN_NAME}_test_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

print(json.dumps(metrics, indent=2))
print('Saved:', metrics_path)

Ultralytics 8.4.84 🚀 Python-3.10.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81090MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,452,814 parameters, 0 gradients, 3.3 GFLOPs
train: /backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced/train... found 24178 images in 14 classes ✅ 
val: /backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced/val... found 5478 images in 14 classes ✅ 
test: /backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced/test... found 5474 images in 14 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 308.2±93.5 MB/s, size: 83.2 KB)
test: Scanning /backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced/test... 5474 images, 0 corrupt: 100% ━━━━━━━━━━━━ 5474/5474 5.1Kit/s 1.1s0.1s
test: New cache created: /backup/Intern/combine_skindiseaseclassifier_devraj/data/yolo_cls_balanced/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 172/172 1.1it/s 2:3

## Write Markdown Summary

In [7]:
summary_path = REPORT_MD_DIR / f'{RUN_NAME}_summary.md'
with summary_path.open('w', encoding='utf-8') as f:
    f.write('# YOLOv8n Classification 12-Epoch Summary\n\n')
    f.write(f'Run folder: `{RUN_DIR}`\n\n')
    f.write(f'Dataset: `{DATA_DIR}`\n\n')
    f.write(f'Model: `{MODEL_FILE}`\n\n')
    f.write(f'Epochs: **{EPOCHS}** fixed, no early stopping.\n\n')
    f.write('## Test Metrics\n\n')
    f.write(f'- Top-1 accuracy: **{metrics["test_top1_acc"]:.4f}**\n')
    f.write(f'- Top-5 accuracy: **{metrics["test_top5_acc"]:.4f}**\n\n')
    f.write('## Key Files\n\n')
    f.write(f'- Best model: `{BEST_MODEL}`\n')
    f.write(f'- Training results: `{RUN_DIR / "results.csv"}`\n')
    f.write(f'- Results plot: `{RUN_DIR / "results.png"}`\n')
    f.write(f'- Confusion matrix: `{RUN_DIR / "confusion_matrix.png"}`\n')

summary_path

PosixPath('/backup/Intern/combine_skindiseaseclassifier_devraj/markdown_reports/yolov8n_cls_fixed_12ep_20260701_121336_summary.md')

## Update Project Structure

In [8]:
subprocess.run(
    ['python3', str(PROJECT_ROOT / 'scripts' / 'update_project_structure.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

Updated: /backup/Intern/combine_skindiseaseclassifier_devraj/PROJECT_STRUCTURE.txt


CompletedProcess(args=['python3', '/backup/Intern/combine_skindiseaseclassifier_devraj/scripts/update_project_structure.py'], returncode=0)